In [ ]:
from typing import List, Optional, Tuple
from uuid import uuid4
import os
import shutil
import torch
import numpy as np
import pandas as pd

import plotly.express as px

from pl_trainer import SBModule, DDPMModule
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks.progress import TQDMProgressBar
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.strategies.ddp import DDPStrategy
from oa_reactdiff.diffusion._schedule import SBSchedule

from oa_reactdiff.trainer.ema import EMACallback
from oa_reactdiff.model import LEFTNet

# Setup

In [ ]:
device = "cuda"
checkpoint_path = "./sb-ts1x.ckpt"
# checkpoint_path = "./ckpt/RPSB-TS1x-All/RGD1xtb-pretrained-1279-leftnet-0-ed8cf581f785/sb-epoch=049-val_ep_scaled_err=0.0433.ckpt"  # RGD1-xtb pretrained

ddpm = SBModule.load_from_checkpoint(
    checkpoint_path=checkpoint_path,
    map_location=device,
)

In [ ]:
ddpm = ddpm.eval()
ddpm = ddpm.to(device)

ddpm.training_config["use_sampler"] = False
ddpm.training_config["swapping_react_prod"] = False

ddpm.setup(stage="fit", device=device, swapping_react_prod=False)

# Generate one bacth

In [ ]:
bz = 48
val_loader = ddpm.val_dataloader(bz=bz, shuffle=False)
ddpm.nfe = 100

it = iter(val_loader)
batch = next(it)
len(val_loader)

In [ ]:
r_pos, ts_pos, p_pos, x0_size, x0_other, rmsds = ddpm.eval_sample_batch(
    batch,
    return_all=True,
)  # 30s for nfe=100

rmsds[0], np.mean(rmsds), np.median(rmsds)

# generate all test reactions

In [ ]:
res, rmsds = ddpm.eval_rmsd(
    val_loader, 
    write_xyz=False,
    bz=bz,
    refpath="ref_ts",
    # localpath="sb-ts1x_ts-v1/ot_ode-10-p1-ip0d5-beta0d3/",
    # max_num_batch=10,
)

In [ ]:
fig = px.histogram(
    x=rmsds,
    marginal="box",
)
fig.show()

In [ ]:
np.mean(rmsds), np.median(rmsds), len(rmsds)